# Lektion 02 - Udforskning af Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** er et samlet framework til at bygge AI-agenter. Det giver en ren, sammensætbar arkitektur med fire kernebyggesten:

- **Client** – forbinder til en AI-modelendepunkt og håndterer kommunikationen
- **Agent** – indkapsler en klient med instruktioner og værktøjsdefinitioner
- **Tools** – udvider agentens evner med brugerdefinerede funktioner, som modellen kan kalde
- **Session** – vedligeholder samtalehistorik for flerfoldige interaktioner

I denne lektion bygger vi en **rejsebookingsagent** der tjekker destinationernes tilgængelighed ved hjælp af disse koncepter.


## Opsætning


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Forståelse af Agent Framework Arkitekturen

Microsoft Agent Framework følger en lagdelt arkitektur:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Klient** – En `FoundryChatClient` forbinder til en Azure OpenAI-udrulning. Den håndterer autentificering, anmodningsformatering og svarfortolkning.
2. **Agent** – Oprettet fra klienten via `provider.create_agent()`, agenten kombinerer modeladgang med instruktioner (systemprompt) og værktøjer.
3. **Værktøjer** – Python-funktioner dekoreret med `@tool`, som agenten kan påkalde for at udføre handlinger eller hente data.
4. **Session** – Et `AgentSession`-objekt (oprettet via `agent.create_session()`), som gemmer samtalehistorik og muliggør flertrins-dialog, hvor agenten husker tidligere kontekst.

Lad os bygge hvert lag trin for trin.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Tilføjelse af værktøjer med @tool-dekorationen

Værktøjer giver agenter mulighed for at udføre handlinger ud over at generere tekst. `@tool`-dekorationen omdanner en almindelig Python-funktion til noget, agenten kan kalde.

Vigtige punkter:
- Brug `Annotated[type, "beskrivelse"]` så modellen forstår hver parameter.
- Docstringen bliver værktøjets beskrivelse, som modellen ser.
- `approval_mode="never_require"` betyder, at værktøjet kører automatisk uden brugerbekræftelse.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Oprettelse af en Agent med Værktøjer

Nu kombinerer vi klienten, instruktionerne og værktøjerne til en agent. `instructions` fungerer som systemprompten — de definerer agentens persona og adfærd.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Fler-sidede samtaler med sessioner

En `AgentSession` (oprettet via `agent.create_session()`) holder styr på alle beskeder i en samtale. Ved at sende den samme session til hvert `agent.run()`-kald får agenten adgang til hele samtalens historik og kan referere tilbage til tidligere beskeder.

Vi sender `tools=[check_destination_availability]` så agenten kan kalde vores tilgængelighedstjekker under hvert trin.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Resumé

I denne lektion udforskede du de fire søjler i Microsoft Agent Framework:

| Koncept | Hvad du lærte |
|---------|------------------|
| **Client** | `FoundryChatClient` opretter forbindelse til Azure OpenAI med legitimationsbaseret godkendelse |
| **Agent** | `provider.create_agent()` samler en modelforbindelse med instruktioner og et navn |
| **Tools** | `@tool`-dekorationen eksponerer Python-funktioner, som agenten kan kalde |
| **Session** | `agent.create_session()` bevarer samtalehistorik på tværs af flere runder |

Disse byggesten kombineres for at skabe agenter, der kan føre naturlige samtaler, kalde eksterne funktioner og bevare kontekst — fundamentet for mere avancerede agent-mønstre i senere lektioner.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
